# Лабораторна робота 10. Завдання 3c. Коти / собаки (2000 / 500 / 500): аугментація + тонке налаштування, ResNet50

Виконав: Бабин Богдан Олегович, група 443-4. Ноутбук 3 з 3 для Завдання 3 (пункти 5–14).

Експерименти (усі з однаковим seed, оцінка на тестовому наборі 1000 фото):
1. **Перенесення навчання без аугментації** — база для порівняння;
2. **Перенесення навчання + аугментація** (параметри з методички);
3. **+ Fine Tuning останнього блоку** (conv5_block3);

Класифікатор — найкращий з лабораторної 9: GlobalAveragePooling2D → Dense 32 → Dropout 0.6 → Dense 1 (sigmoid), Adam 1e-4.

**Перед запуском:** Среда выполнения → T4 GPU → «Выполнить все».

In [ ]:
import os, shutil, time, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Flatten, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
%matplotlib inline
tf.keras.utils.set_random_seed(42)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 5–6. Обмежений набір даних і структура каталогів

In [ ]:
!wget -q -O cats_dogs.zip https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
!unzip -q -o cats_dogs.zip

In [ ]:
%%writefile prepare_dataset.py
"""
prepare_dataset.py — обмежений набір Dogs vs Cats для лабораторної 10 (змінений скрипт з лабораторної 7).
Відкидає пошкоджені файли, перемішує і копіює БЕЗ ПОВТОРІВ:
  train: по 2000 котів і собак, val: по 500, test: по 500   →   dataset/{train,val,test}/{cats,dogs}
"""
import os, sys, shutil, random
from PIL import Image
N_TRAIN, N_VAL, N_TEST = 2000, 500, 500
random.seed(42)

def is_valid(path):
    try:
        if os.path.getsize(path) == 0:
            return False
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            img.convert('RGB').load()
        return True
    except Exception:
        return False

src = {'cats': 'PetImages/Cat', 'dogs': 'PetImages/Dog'}
for cls, d in src.items():
    files = sorted(f for f in os.listdir(d) if f.lower().endswith('.jpg'))
    random.shuffle(files)
    chosen = []
    for f in files:                                   # беремо перші 3000 справних фото
        if is_valid(os.path.join(d, f)):
            chosen.append(f)
        if len(chosen) == N_TRAIN + N_VAL + N_TEST:
            break
    parts = {'train': chosen[:N_TRAIN], 'val': chosen[N_TRAIN:N_TRAIN + N_VAL], 'test': chosen[N_TRAIN + N_VAL:]}
    for part, fs in parts.items():
        out = os.path.join('dataset', part, cls)
        shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
        for i, f in enumerate(fs):
            shutil.copy2(os.path.join(d, f), os.path.join(out, f'{cls[:-1]}.{i}.jpg'))
    assert len(set(chosen)) == len(chosen)           # усі фото різні
for part in ('train', 'val', 'test'):
    print(part, {c: len(os.listdir(os.path.join('dataset', part, c))) for c in ('cats', 'dogs')})

In [ ]:
!python prepare_dataset.py
for root, dirs, fs in sorted(os.walk('dataset')):
    print('    ' * root.count(os.sep) + os.path.basename(root) + '/' + (f'   ({len(fs)} файлів)' if fs else ''))

## 7–9. Генератори
Навчальний генератор з аугментацією — параметри з методички (rotation 40, зсуви 0.2, zoom 0.2, horizontal flip, fill nearest).
Замість `rescale=1./255` — `preprocess_input` попередньо навченої мережі (так само, як при її навчанні на ImageNet; див. лаби 8–9).
Генератор для перевірки і тестування — без доповнення.

In [ ]:
size = 150
AUG = dict(rotation_range=40, width_shift_range=0.2, height_shift_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest')
AUG_LIGHT = dict(rotation_range=15, width_shift_range=0.1, height_shift_range=0.1, zoom_range=0.1, horizontal_flip=True, fill_mode='nearest')

def flow(part, aug=None, shuffle=True):
    gen = ImageDataGenerator(preprocessing_function=preprocess_input, **(aug or {}))
    return gen.flow_from_directory('dataset/' + part, target_size=(size, size), batch_size=32, class_mode='binary', shuffle=shuffle)

train_plain = flow('train')
train_aug = flow('train', AUG)
val_gen, test_gen = flow('val', shuffle=False), flow('test', shuffle=False)

# як виглядає аугментація: одне фото, 8 випадкових варіантів
show = ImageDataGenerator(**AUG).flow(img_to_array(load_img('dataset/train/cats/cat.0.jpg', target_size=(size, size)))[None], batch_size=1)
plt.figure(figsize=(16, 2.4))
plt.subplot(1, 9, 1); plt.imshow(load_img('dataset/train/cats/cat.0.jpg', target_size=(size, size))); plt.title('оригінал'); plt.axis('off')
for j in range(8):
    plt.subplot(1, 9, j + 2); plt.imshow(next(show)[0].astype('uint8')); plt.axis('off')
plt.suptitle('Data Augmentation: одне навчальне фото → нові варіанти щоепохи'); plt.tight_layout(); plt.show()

## 10. Попередньо навчена ResNet50 і складена мережа

In [ ]:
base_net = ResNet50(weights='imagenet', include_top=False, input_shape=(size, size, 3))
names = [l.name for l in base_net.layers]
CUTS = {1: 'conv5_block2_out', 2: 'conv5_block1_out'}   # fine tuning: розморожуються шари ПІСЛЯ цього шару

def unfreeze(k):
    base_net.trainable = k > 0
    cut = names.index(CUTS[k]) if k else len(names)
    for j, layer in enumerate(base_net.layers):
        layer.trainable = j > cut and not isinstance(layer, BatchNormalization)

def new_model():
    tf.keras.utils.set_random_seed(42)
    unfreeze(0)
    model = Sequential([Input(shape=(size, size, 3)), base_net, GlobalAveragePooling2D(),
                        Dense(32, activation='relu'), Dropout(0.6), Dense(1, activation='sigmoid')])
    model.compile(loss='binary_crossentropy', optimizer=Adam(1e-4), metrics=['accuracy'])
    return model

model = new_model()
model.summary()
init_w = model.get_weights()          # початкові ваги класифікатора — щоб усі експерименти стартували однаково

In [ ]:
def plot_history(h, title):
    ep = range(1, len(h['loss']) + 1)
    plt.figure(figsize=(13, 4))
    plt.subplot(1, 2, 1); plt.plot(ep, h['accuracy'], 'b-o', ms=3, label='Навчальні'); plt.plot(ep, h['val_accuracy'], 'r-o', ms=3, label='Перевірочні')
    plt.title('Точність'); plt.xlabel('Епоха'); plt.legend(); plt.grid(alpha=.3)
    plt.subplot(1, 2, 2); plt.plot(ep, h['loss'], 'b-o', ms=3, label='Навчальні'); plt.plot(ep, h['val_loss'], 'r-o', ms=3, label='Перевірочні')
    plt.title('Помилка (loss)'); plt.xlabel('Епоха'); plt.legend(); plt.grid(alpha=.3)
    plt.suptitle(title); plt.tight_layout(); plt.show()

results, hists, weights = [], {}, {}
def run(key, name, train_gen, epochs, lr=1e-4, blocks=0, start=None, patience=None):
    if start is not None:
        unfreeze(0); model.set_weights(start)        # усі стартові ваги збережені при замороженій мережі
    unfreeze(blocks)
    model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=lr), metrics=['accuracy'])
    n_tr = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    cb = [EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)] if patience else []
    tf.keras.utils.set_random_seed(42); t0 = time.time()
    h = model.fit(train_gen, epochs=epochs, validation_data=val_gen, verbose=2, callbacks=cb).history
    t = time.time() - t0
    loss, acc = model.evaluate(test_gen, verbose=0)
    results.append({'експеримент': f'{key}. {name}', 'навчаються параметрів': n_tr, 'епох': len(h['loss']),
                    'train acc, %': round(h['accuracy'][-1] * 100, 2), 'val acc, %': round(h['val_accuracy'][-1] * 100, 2),
                    'розрив train−val, %': round((h['accuracy'][-1] - h['val_accuracy'][-1]) * 100, 2),
                    'val loss': round(h['val_loss'][-1], 4), 'test acc, %': round(acc * 100, 2), 'test loss': round(loss, 4), 'час, с': round(t)})
    hists[key], weights[key] = h, model.get_weights()
    plot_history(h, f'ResNet50, {key}. {name} — test acc {acc*100:.2f}%')
    return acc

### Експеримент 1. Перенесення навчання без аугментації (база)

In [ ]:
run('1', 'перенесення навчання, без аугментації', train_plain, epochs=10, start=init_w)

### Експеримент 2. Перенесення навчання + аугментація

In [ ]:
run('2', 'перенесення навчання + аугментація', train_aug, epochs=15, start=init_w)

## 11. Fine Tuning
Стартуємо з ваг експерименту 2 (класифікатор уже навчений), розморожуємо **останній блок (conv5_block3)**,
донавчаємо з аугментацією і малою швидкістю Adam 1e-5; EarlyStopping (patience 3) повертає ваги найкращої епохи.

In [ ]:
run('3', 'аугментація + fine tuning 1 блоку', train_aug, epochs=10, lr=1e-5, blocks=1, start=weights['2'], patience=3)

## 13. Порівняння і найкраща модель

In [ ]:
res = pd.DataFrame(results)
res.insert(0, 'модель', 'ResNet50')
res.to_csv('lab10_3_ResNet50.csv', index=False)
res

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for k, h in hists.items():
    ax[0].plot(range(1, len(h['val_accuracy']) + 1), h['val_accuracy'], '-o', ms=3, label=k)
    ax[1].plot(range(1, len(h['accuracy']) + 1), np.array(h['accuracy']) - np.array(h['val_accuracy']), '-o', ms=3, label=k)
ax[0].set_title('Точність на перевірочних даних'); ax[1].set_title('Розрив train acc − val acc (перенавчання)')
ax[1].axhline(0, color='k', lw=.5)
for a_ in ax: a_.set_xlabel('Епоха'); a_.legend(); a_.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
best_i = int(res['test acc, %'].idxmax()); best_key = res.loc[best_i, 'експеримент'].split('.')[0]
blocks = {'3': 1, '4': 2}.get(best_key, 0)
unfreeze(blocks); model.set_weights(weights[best_key])
model.compile(loss='binary_crossentropy', optimizer=Adam(1e-5), metrics=['accuracy'])
best_model = model
print('Найкраща модель:', res.loc[best_i, 'експеримент'], '| test acc =', res.loc[best_i, 'test acc, %'], '%')
best_model.save('lab10_3_ResNet50_best.keras')

### Розпізнавання власних фото (ті самі, що в лабах 7–9)

In [ ]:
os.makedirs('test_images', exist_ok=True)
# Власні фото з лаб 7–9 (lab8_test_images.zip), якщо архів завантажено через панель «Файли».
# Інакше — 20 нових фото котів і собак з інтернету (фотобанк pixel-shot.com), яких немає в наборі Dogs vs Cats.
if os.path.exists('lab8_test_images.zip'):
    zipfile.ZipFile('lab8_test_images.zip').extractall('test_images')
else:
    import re, io, requests
    from PIL import Image
    P = 'https://pixel-shot.com/en/showcase/'
    WEB = [('cat_1','i-7260473-0/cute-funny-cat-at-home'), ('cat_2','i-913877-0/cute-little-kittens'), ('cat_3','i-2373827-0/portrait-of-small-cute-kitten'),
           ('cat_4','i-5258613-0/cute-little-kitten-at-home'), ('cat_5','i-1802418-0/cute-gray-kitten-at-home'), ('cat_6','i-1613036-0/cute-little-kitten-on-light-background'),
           ('cat_7','i-2374755-0/small-cute-kitten-on-carpet'), ('cat_8','i-1201688-0/cute-little-kitten-outdoors'), ('cat_9','i-6094608-0/cute-funny-kitten-at-home'),
           ('cat_10','i-913955-0/cute-mother-cat-and-little-kitten'),
           ('dog_1','i-4587052-0/cute-funny-dog-outdoors'), ('dog_2','i-1003255-0/funny-cute-dog--outdoors'), ('dog_3','i-5166545-0/cute-funny-dog-outdoors'),
           ('dog_4','i-3770385-0/cute-purebred-dog-outdoors'), ('dog_5','i-8371437-0/cute-labrador-outdoors-on-summer-day'), ('dog_6','i-2468995-0/cute-dog-lying-outdoor'),
           ('dog_7','i-4153272-0/cute-lovely-dog-outdoors-on-winter-day'), ('dog_8','i-8397956-0/cute-labrador-dog-in-training-area-outdoors--closeup'),
           ('dog_9','i-7397977-0/cute-dog-on-green-lawn-in-park'), ('dog_10','i-8371250-0/cute-labrador-in-park-on-summer-day')]
    UA = {'User-Agent': 'Mozilla/5.0 (student lab project; ChNU)'}
    for name, page in WEB:
        try:
            html = requests.get(P + page, headers=UA, timeout=20).text
            url = re.search(r'https://img\.pixel-shot\.com/[^"\'\s]+?showcase\.(?:webp|jpg|jpeg|png)', html).group(0)
            Image.open(io.BytesIO(requests.get(url, headers=UA, timeout=30).content)).convert('RGB').save(f'test_images/{name}.jpg', quality=92)
        except Exception as e:
            print('пропущено', name, e)
paths = sorted(os.path.join(r, f) for r, _, fs in os.walk('test_images') for f in fs
               if f.lower().endswith(('.jpg', '.jpeg', '.png')) and not f.startswith('._'))
print(len(paths), 'зображень')

In [ ]:
def predict_images(paths, title):
    rows = []; cols = 6; rn = int(np.ceil(len(paths) / cols))
    plt.figure(figsize=(cols * 2.8, rn * 3.1))
    for j, p in enumerate(paths):
        x = preprocess_input(np.expand_dims(img_to_array(load_img(p, target_size=(size, size))), 0))
        prob = float(best_model.predict(x, verbose=0)[0][0])
        ans = 'собака' if prob > 0.5 else 'кіт'
        truth = 'кіт' if os.path.basename(p).lower().startswith('cat') else 'собака'
        conf = prob if prob > 0.5 else 1 - prob
        plt.subplot(rn, cols, j + 1); plt.imshow(load_img(p)); plt.axis('off')
        plt.title(f'{ans} ({conf*100:.0f}%)\n{os.path.basename(p)}', fontsize=8, color='green' if ans == truth else 'red')
        rows.append({'файл': os.path.basename(p), 'відповідь': ans, 'впевненість, %': round(conf * 100, 1), 'вірно': ans == truth})
    plt.suptitle(title); plt.tight_layout(); plt.show()
    df = pd.DataFrame(rows); print(f'{title}: вірно {int(df["вірно"].sum())} з {len(df)}'); return df

normal = [p for p in paths if '_h' not in os.path.basename(p)]
hard = [p for p in paths if '_h' in os.path.basename(p) and '_crop' not in os.path.basename(p)]
own = predict_images(normal, 'ResNet50 (лаба 10): звичайні зображення')
own_hard = predict_images(hard, 'ResNet50 (лаба 10): тварина в траві') if hard else pd.DataFrame({'файл': [], 'вірно': []})
pd.concat([own, own_hard]).to_csv('lab10_3_ResNet50_own.csv', index=False)

In [ ]:
cmp = pd.DataFrame([{'робота': 'лаба 7: власна CNN з нуля (17,5 тис. фото)', 'test acc, %': 84.75, 'власні фото': '16/20, 3/6'}] +
    [{'робота': f'{k}: ResNet50', 'test acc, %': v, 'власні фото': {'лаба 8': '19/20, 6/6', 'лаба 9': '19/20, 6/6'}[k]} for k, v in {'лаба 8': 97.33, 'лаба 9': 97.78}.items()] +
    [{'робота': f'лаба 10: ResNet50, ' + r['експеримент'], 'test acc, %': r['test acc, %'],
      'власні фото': f"{int(own['вірно'].sum())}/{len(own)}" + (f", {int(own_hard['вірно'].sum())}/{len(own_hard)}" if len(own_hard) else ' (нові фото з інтернету)') if r['експеримент'] == res.loc[best_i, 'експеримент'] else ''}
     for _, r in res.iterrows()])
cmp

In [ ]:
from google.colab import files
for f in ['lab10_3_ResNet50_best.keras', 'lab10_3_ResNet50.csv', 'lab10_3_ResNet50_own.csv']:
    files.download(f)